In [2]:
# load libraries 
import pandas as pd 
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pypq
import pyarrow.csv as pycsv
import textwrap 
from time import time 
import plotly.io as pio
import os


tqdm.pandas()
plt.rcParams.update({'font.size': 22})
sns.set(style="ticks", context="talk")
plt.style.use("dark_background")
pd.options.plotting.backend = 'plotly'
pio.templates.default = 'plotly_dark+presentation'

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [2]:
# some helper functions 
def read_parquet(path, engine='pyarrow', columns=None, convert_dtypes=True, **args):
    """
    Read a parquet file (or a directory of parquet files) 
    columns: list of columns to read, by default, read all columns
    convert_dtypes: if True, convert datatypes to save RAM (takes extra time)
    """
    
    path = Path(path)
    name = path.stem 
    column_st = 'columns="all"' if columns is None else f'{columns=!r}'
    print(f'\nReading {column_st} from {path!r} using {engine=!r}.')

    tic = time()
    df = pd.read_parquet(path, engine=engine, columns=columns, **args)
    toc = time()
    print(f'Read {len(df):,} rows from {path.stem!r} in {toc-tic:.2f} sec.')
    
    if convert_dtypes:
        tic = time()
        size_before = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024

        string_cols_d = {}
        for col, dtype in df.dtypes.to_dict().items():
            if dtype == 'object':  # convert object columns to string
                string_cols_d[col] = 'string[python]'
            if col == 'type' or col == 'concept_name':
                if dtype != 'category':
                    string_cols_d[col] = 'category'
            if col == 'publication_month':
                if dtype != 'uint8':
                    string_cols_d[col] = 'uint8'
            if col == 'score':
                if dtype != 'float16':
                    string_cols_d[col] = 'float16'
        # print(f'{string_cols_d=}')
        df = df.astype(string_cols_d) 
        
        size_after = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024
        toc = time()
        print(f'Converting dtypes took {toc-tic:.2f} sec. Size before: {size_before:.2f}GB, after: {size_after:.2f}GB.')
    
    display('Top 3 rows:', df.head(3))
    return df


def peek_parquet(path):
    """
    peeks at a parquet file (or a directory containing parquet files) without reading the whole thing and prints the following:
    * Path
    * schema
    * number of pieces (fragments)
    * number of rows 
    """

    path = Path(path)
    parq_file = pypq.ParquetDataset(path)
    piece_count = len(parq_file.fragments)
    schema = textwrap.indent(parq_file.schema.to_string(), ' '*4)
    row_count = sum(frag.count_rows() for frag in parq_file.fragments)
    if Path(path).is_dir():
      size = sum(Path(frag.path).stat().st_size for frag in parq_file.fragments)
    else:
      size = path.stat().st_size
    
    st = [
        f'Name: {path.stem!r}',  
        f'Path: {str(path)!r}',
        f'Size: {size/1024/1024/1024:.2g} GB',
        f'Files: {piece_count:,}',
        f'Rows: {row_count:,}',
        f'Schema:\n{schema}',
        f'5 random rows:',
    ]
    print('\n'.join(st))
    sample_df = parq_file.fragments[0].head(5).to_pandas()  # read 5 rows from the first fragment
    display(sample_df)

    return

def read_smaller_tables(name):
    """
    Some smaller tables exist as a CSV only
    """
    assert name in ['institutions', 'institutions_geo', 'concepts']
    path = basepath / 'csv-files'/ month / name
    df = pd.read_csv(f'{path}.csv.gz', engine='c')
    return df

def tsv_to_parquet(tsv_path, output_filename=None, dtype=None, chunksize=100000, **read_csv_kwargs):
    """
    Convert a TSV file to Parquet format and save it in the current directory.
     
    Args:
        tsv_path: Path to the TSV file to convert
        output_filename: Optional output filename (defaults to same name as TSV but .parquet)
        dtype: Optional dict of column dtypes for reading TSV
        **read_csv_kwargs: Additional arguments to pass to pd.read_csv
    
    Returns:
        Path to the created parquet file
    """

    #get the output path
    notebook_dir = Path.cwd()
    if output_filename is None:
        output_filename = Path(tsv_path).stem + '.parquet'
    else: 
        output_path = notebook_dir / output_filename

    if not output_path.exists():
        print(f"Converting {tsv_path} to Parquet...")
        
        chunks = pd.read_csv(
            tsv_path,
            sep = '\t',
            dtype=dtype,
            low_memory=False,
            chunksize=chunksize,
            **read_csv_kwargs
        )

        for i, chunk in enumerate(chunks):
            chunk.to_parquet(
                output_path,
                index=False,
                engine='fastparquet',
                append=(i != 0)
            )
            
            if (i + 1) % 10 == 0:
                print(f"Processed {(i + 1) * chunksize} rows...")


        print(f"Successfully wrote Parquet file to: {output_path}")

    return output_path        

In [8]:
def tsv_to_parquet_pyarrow(tsv_path, output_filename=None, column_types=None):
    """
    Convert a TSV file to Parquet format using PyArrow directly.
    
    Args:
        tsv_path: Path to the TSV file to convert
        output_filename: Optional output filename (defaults to same name as TSV but .parquet)
        column_types: Optional dict of column types (e.g., {'patent_id': pa.string()})
    
    Returns:
        Path to the created parquet file
    """
    notebook_dir = Path.cwd()
    if output_filename is None:
        output_filename = Path(tsv_path).stem + '.parquet'
    output_path = notebook_dir / output_filename
    
    if not output_path.exists():
        print(f"Converting {tsv_path} to Parquet...")
        
        # Set up parse options for TSV
        parse_options = pycsv.ParseOptions(delimiter='\t')
        
        # Set up convert options with column types if provided
        convert_options = None
        if column_types:
            convert_options = pycsv.ConvertOptions(column_types=column_types, timestamp_parsers=["%Y-%m-%d"])
        
        # Read TSV directly into Arrow table (streams data, doesn't load all into memory)
        reader = pycsv.open_csv(
            tsv_path,
            parse_options=parse_options,
            convert_options=convert_options
        )

        writer = None
        for batch in reader:
            if writer is None:
                writer = pypq.ParquetWriter(output_path, batch.schema)
            writer.write_batch(batch)
        if writer:
            writer.close()
             
    
    return output_path

In [9]:
USPTOPath = '/data/shared/USPTO-patents/'

'''assignee_output_path = tsv_to_parquet('/data/shared/USPTO-patents/g_assignee_disambiguated.tsv', dtype={'patent_id': str})
location_op = tsv_to_parquet('/data/shared/USPTO-patents/g_location_disambiguated.tsv', dtype={'patent_id': str})
g_us_patent_citation_op = tsv_to_parquet('/data/shared/USPTO-patents/g_us_patent_citation.tsv', dtype={'patent_id': str})
foreign_citations_op = tsv_to_parquet('/data/shared/USPTO-patents/g_foreign_citation.tsv', dtype={'patent_id': str, 'citation_application_id': str})
inventor_op = tsv_to_parquet_pyarrow(USPTOPath+'g_inventor_disambiguated.tsv', column_types={'patent_id': pa.string()})
application_op = tsv_to_parquet_pyarrow(USPTOPath+'g_us_application_citation.tsv', column_types={'patent_id': pa.string()}) '''
##for tsv in USPTOpath:
   ## tsv+'_op' = tsv_to_parquet(USPTOPath+tsv, column_types={'patent_id': pa.string()})

##tsv_files = [f for f in os.listdir(USPTOPath) if f.endswith('.tsv')]
tsv_files = list(Path(USPTOPath).glob('*.tsv'))

# Convert all TSV files (function will skip if parquet already exists)
for tsv_file in tsv_files:
    ##tsv_file = Path(tsv_file)
    print(tsv_file.stem)
    if tsv_file.stem in ['g_us_application_citation', 'g_us_patent_citation']:
        continue
    tsv_to_parquet_pyarrow(
        tsv_file,
        column_types={'patent_id': pa.string(), 'patent_date': pa.timestamp("s")} #pa.dictionary() takes 2 values takes the key string and the second part would be the number of unique values there are (make numeric mapping based on each unique category
    )

print("All conversions complete!")

g_us_application_citation
g_us_application_citation_2
g_wipo_technology
g_us_patent_citation_5
g_patent
g_gov_interest_contracts
g_location_disambiguated
g_us_application_citation_4
g_gov_interest_org
g_attorney_disambiguated
g_us_patent_citation_6
Converting /data/shared/USPTO-patents/g_us_patent_citation_6.tsv to Parquet...
g_us_patent_citation_2
Converting /data/shared/USPTO-patents/g_us_patent_citation_2.tsv to Parquet...
g_us_application_citation_3
g_us_patent_citation_1
Converting /data/shared/USPTO-patents/g_us_patent_citation_1.tsv to Parquet...
g_us_application_citation_5
g_foreign_citation
g_inventor_disambiguated
g_uspc_at_issue
g_assignee_disambiguated
g_us_patent_citation_3
Converting /data/shared/USPTO-patents/g_us_patent_citation_3.tsv to Parquet...


ArrowInvalid: In CSV column #5: CSV conversion error to null: invalid value 'A'

In [3]:
def split_tsv_sequential(input_file, num_splits=5):
    """Split TSV into 5 sequential chunks with _1, _2, _3, _4, _5 naming"""
    
    # Get base filename without extension
    base_name = input_file.replace('.tsv', '')
    
    # First pass: count total rows
    print("Counting rows...")
    with open(input_file, 'r', encoding='utf-8') as f:
        header = f.readline()
        total_rows = sum(1 for _ in f)
    
    print(f"Total rows: {total_rows:,}")
    
    rows_per_file = total_rows // num_splits
    print(f"Rows per file: ~{rows_per_file:,}")
    
    # Second pass: split the file
    with open(input_file, 'r', encoding='utf-8') as f:
        header = f.readline()
        
        file_num = 1
        row_count = 0
        output = open(f'{base_name}_{file_num}.tsv', 'w', encoding='utf-8')
        output.write(header)  # Write header to first file
        
        for line in f:
            output.write(line)
            row_count += 1
            
            # Start new file when we hit the threshold
            if row_count >= rows_per_file and file_num < num_splits:
                output.close()
                print(f"Created {base_name}_{file_num}.tsv with {row_count:,} rows")
                
                file_num += 1
                row_count = 0
                output = open(f'{base_name}_{file_num}.tsv', 'w', encoding='utf-8')
                output.write(header)  # Write header to each file
        
        output.close()
        print(f"Created {base_name}_{file_num}.tsv with {row_count:,} rows")
        print("\nDone! Created files:")
        for i in range(1, num_splits + 1):
            print(f"  - {base_name}_{i}.tsv")

# Usage - replace 'your_file.tsv' with your actual filename
split_tsv_sequential('/data/shared/USPTO-patents/g_us_patent_citation.tsv', num_splits=6)

Counting rows...
Total rows: 151,140,729
Rows per file: ~25,190,121
Created /data/shared/USPTO-patents/g_us_patent_citation_1.tsv with 25,190,121 rows
Created /data/shared/USPTO-patents/g_us_patent_citation_2.tsv with 25,190,121 rows
Created /data/shared/USPTO-patents/g_us_patent_citation_3.tsv with 25,190,121 rows
Created /data/shared/USPTO-patents/g_us_patent_citation_4.tsv with 25,190,121 rows
Created /data/shared/USPTO-patents/g_us_patent_citation_5.tsv with 25,190,121 rows
Created /data/shared/USPTO-patents/g_us_patent_citation_6.tsv with 25,190,124 rows

Done! Created files:
  - /data/shared/USPTO-patents/g_us_patent_citation_1.tsv
  - /data/shared/USPTO-patents/g_us_patent_citation_2.tsv
  - /data/shared/USPTO-patents/g_us_patent_citation_3.tsv
  - /data/shared/USPTO-patents/g_us_patent_citation_4.tsv
  - /data/shared/USPTO-patents/g_us_patent_citation_5.tsv
  - /data/shared/USPTO-patents/g_us_patent_citation_6.tsv


In [5]:
peek_parquet("g_foreign_citation.parquet")

Name: 'g_foreign_citation'
Path: 'g_foreign_citation.parquet'
Size: 0.84 GB
Files: 1
Rows: 45,007,862
Schema:
    patent_id: string
    citation_sequence: int64
    citation_application_id: string
    citation_date: date32[day]
    citation_category: string
    citation_country: string
5 random rows:


,patent_id,citation_sequence,citation_application_id,citation_date,citation_category,citation_country
0,9872910,4,2013-512894,2013-04-01,cited by applicant,JP
1,7580255,0,1 104 150,2001-05-01,cited by other,EP
2,10221436,84,WO-2007138178,2007-12-01,cited by applicant,WO
3,5928533,1,2641718,1990-07-01,,FR
4,11198379,11,2013-189029,2013-09-01,cited by applicant,JP


In [1]:
import pandas as pd
USPTOPath = '/data/shared/USPTO-patents/'
df = pd.read_csv(USPTOPath+'g_us_application_citation_3.tsv', sep = '\t', usecols=['patent_id', 'wipo_kind', 'citation_document_number'])
cols = ['wipo_kind', 'citation_document_number']
df[df[cols].isna().any(axis=1)]

/tmp/ipykernel_2500626/3875476092.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(USPTOPath+'g_us_application_citation_3.tsv', sep = '\t', usecols=['patent_id', 'wipo_kind', 'citation_document_number'])


,patent_id,citation_document_number,wipo_kind
7241,11619658,20140974860,NaN
24436,11109244,20090376585,NaN
47019,9857167,20107018294,NaN
50365,7481776,20061051342,NaN
63113,8244613,20930196204,NaN
...,...,...,...
15391853,10720322,20003049372,NaN
15401525,11702888,20182009233,NaN
15404711,8235724,20050700809,NaN
15425039,11471682,20031200323,NaN


In [ ]:
 tsv_to_parquet_pyarrow(
        USPTOPath + "g_cpc_current.tsv",
        column_types={'patent_id': pa.string(), 'patent_date': pa.timestamp("s")}